# Payoff Diagrams 
Dayanni Godoy Rosales


In [ ]:
import yfinance as yf
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd


## 1. Obtención de precios reales (yfinance)

In [ ]:
tk  = yf.Ticker('SPY')
S0  = tk.fast_info['last_price']
exp = '2026-04-17'
K   = 680

calls = tk.option_chain(exp).calls
puts  = tk.option_chain(exp).puts

row_call = calls[calls['strike'] == K].iloc[0]
row_put  = puts[puts['strike'] == K].iloc[0]

C = (row_call['bid'] + row_call['ask']) / 2
P = (row_put['bid']  + row_put['ask'])  / 2

print(f"S0 = {S0:.2f}  |  K = {K}  |  C (call mid) = {C:.4f}  |  P (put mid) = {P:.4f}")
print(f"Vencimiento: {exp}")


## 2. Cálculo de payoffs brutos y P&L neto

In [ ]:
# Rango de precios al vencimiento (dinámico alrededor de S0 y K)
lower = min(S0, K) - 80
upper = max(S0, K) + 80
S_range = np.linspace(lower, upper, 1000)

#Payoff bruto 
call_long_bruto  =  np.maximum(S_range - K, 0)
call_short_bruto = -np.maximum(S_range - K, 0)
put_long_bruto   =  np.maximum(K - S_range, 0)
put_short_bruto  = -np.maximum(K - S_range, 0)

# P&L neto (descontando prima) 
pnl_call_long  = call_long_bruto  - C
pnl_call_short = call_short_bruto + C
pnl_put_long   = put_long_bruto   - P
pnl_put_short  = put_short_bruto  + P


## 3. Las cuatro posiciones básicas (P&L neto, 2×2)

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(13, 9))

configs = [
    (axes[0, 0], pnl_call_long,  f'Long Call  | K={K} | C={C:.2f}',  'steelblue'),
    (axes[0, 1], pnl_call_short, f'Short Call | K={K} | C={C:.2f}',  'tomato'),
    (axes[1, 0], pnl_put_long,   f'Long Put   | K={K} | P={P:.2f}',  'seagreen'),
    (axes[1, 1], pnl_put_short,  f'Short Put  | K={K} | P={P:.2f}',  'darkorange'),
]

for ax, pnl, title, color in configs:
    ax.plot(S_range, pnl, color=color, linewidth=2)
    ax.axhline(0, color='black', linewidth=0.8, linestyle='--')
    ax.axvline(K,  color='gray',  linewidth=0.8, linestyle=':',  label=f'Strike K={K}')
    ax.axvline(S0, color='navy', linewidth=1.2, linestyle='-.', label=f'S₀={S0:.1f}')
    ax.fill_between(S_range, pnl, 0, where=(pnl > 0), alpha=0.15, color='green', label='Ganancia')
    ax.fill_between(S_range, pnl, 0, where=(pnl < 0), alpha=0.15, color='red',   label='Pérdida')
    ax.set_title(title, fontsize=11, fontweight='bold')
    ax.set_xlabel('Precio al vencimiento $S_T$ ($)')
    ax.set_ylabel('P&L ($)')
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

fig.suptitle(f'Las cuatro posiciones básicas — SPY | S₀={S0:.2f} | K={K} | Exp {exp}',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()


## 4. Comparación payoff bruto vs. P&L neto (Call y Put por separado)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

#  Long Call 
be_call = K + C
axes[0].plot(S_range, call_long_bruto, color='steelblue',
             linewidth=2, linestyle='--', label='Payoff bruto')
axes[0].plot(S_range, pnl_call_long,   color='steelblue',
             linewidth=2, label='P&L neto')
axes[0].axhline(0, color='black', linewidth=0.8, linestyle='--')
axes[0].axvline(K,       color='gray',  linewidth=0.8, linestyle=':',  label=f'Strike K={K}')
axes[0].axvline(be_call, color='green', linewidth=1.2, linestyle=':',  label=f'Break-even={be_call:.2f}')
axes[0].axvline(S0,      color='navy',  linewidth=1.2, linestyle='-.', label=f'S₀={S0:.1f}')
axes[0].fill_between(S_range, pnl_call_long, 0, where=(pnl_call_long > 0), alpha=0.15, color='green')
axes[0].fill_between(S_range, pnl_call_long, 0, where=(pnl_call_long < 0), alpha=0.15, color='red')
axes[0].set_title('Long Call: bruto vs. neto', fontweight='bold')
axes[0].set_xlabel('$S_T$ ($)')
axes[0].set_ylabel('Valor ($)')
axes[0].legend(fontsize=8)
axes[0].grid(True, alpha=0.3)

#  Long Put 
be_put = K - P
axes[1].plot(S_range, put_long_bruto, color='seagreen',
             linewidth=2, linestyle='--', label='Payoff bruto')
axes[1].plot(S_range, pnl_put_long,   color='seagreen',
             linewidth=2, label='P&L neto')
axes[1].axhline(0, color='black', linewidth=0.8, linestyle='--')
axes[1].axvline(K,      color='gray',  linewidth=0.8, linestyle=':',  label=f'Strike K={K}')
axes[1].axvline(be_put, color='green', linewidth=1.2, linestyle=':',  label=f'Break-even={be_put:.2f}')
axes[1].axvline(S0,     color='navy',  linewidth=1.2, linestyle='-.', label=f'S₀={S0:.1f}')
axes[1].fill_between(S_range, pnl_put_long, 0, where=(pnl_put_long > 0), alpha=0.15, color='green')
axes[1].fill_between(S_range, pnl_put_long, 0, where=(pnl_put_long < 0), alpha=0.15, color='red')
axes[1].set_title('Long Put: bruto vs. neto', fontweight='bold')
axes[1].set_xlabel('$S_T$ ($)')
axes[1].set_ylabel('Valor ($)')
axes[1].legend(fontsize=8)
axes[1].grid(True, alpha=0.3)

fig.suptitle(f'Payoff bruto vs. P&L neto — SPY | S₀={S0:.2f} | K={K}',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()


## 5. Tabla de Break-Evens de las cuatro posiciones

In [ ]:
#  Break-evens 
be_call = K + C   # Long Call y Short Call comparten el mismo BE de precio
be_put  = K - P   # Long Put  y Short Put  comparten el mismo BE de precio

data = {
    'Posición':         ['Long Call', 'Short Call', 'Long Put', 'Short Put'],
    'Fórmula':          ['K + C',     'K + C',      'K − P',    'K − P'],
    'Break-Even ($)':   [be_call, be_call, be_put, be_put],
    'S₀ ($)':           [S0, S0, S0, S0],
    '|BE − S₀| ($)':    [abs(be_call - S0), abs(be_call - S0),
                          abs(be_put  - S0), abs(be_put  - S0)],
    'Pérdida máx.':     [f'${C:.2f} (prima)',
                          'Ilimitada (↑)',
                          f'${P:.2f} (prima)',
                          f'${K - P:.2f} (si S→0)'],
    'Ganancia máx.':    ['Ilimitada (↑)',
                          f'${C:.2f} (prima)',
                          f'${K - P:.2f} (si S→0)',
                          f'${P:.2f} (prima)'],
}

df_be = pd.DataFrame(data)
df_be = df_be.sort_values('|BE − S₀| ($)').reset_index(drop=True)

# Redondear columnas numéricas para presentación
for col in ['Break-Even ($)', 'S₀ ($)', '|BE − S₀| ($)']:
    df_be[col] = df_be[col].map('{:.2f}'.format)

print("\n=== Tabla de Break-Evens (ordenada por proximidad a S₀) ===\n")
print(df_be.to_string(index=False))
print(f"\n  K = {K}  |  C = {C:.4f}  |  P = {P:.4f}  |  S₀ = {S0:.2f}")
print(f"  BE Long/Short Call = {be_call:.2f}")
print(f"  BE Long/Short Put  = {be_put:.2f}")


---
## 6. Preguntas de análisis


### Pregunta 1  
**¿Cuál de las cuatro posiciones básicas tiene el break-even más cercano al precio actual? ¿Qué implica eso?**

---

La tabla anterior ordena las cuatro posiciones por su distancia `|BE − S₀|`.  
Veamos el razonamiento general:

| Posición | Break-even | Distancia a S₀ |
|---|---|---|
| Long Call / Short Call | `K + C` | El precio debe **subir** `C` dólares por encima del strike |
| Long Put / Short Put | `K − P` | El precio debe **caer** `P` dólares por debajo del strike |

**Conclusión con los datos reales:**  
- La posición con break-even más cercano a S₀ es aquella cuyo break-even requiere **menor movimiento del subyacente** para alcanzarse.  
- Si `|K + C − S₀| < |K − P − S₀|`, el call tiene su BE más cerca; de lo contrario, el put.  
- **Implicación:** Un break-even muy cercano a S₀ significa que la posición empezará a generar P&L positivo con un movimiento pequeño del mercado (para posiciones largas) o que el riesgo de incurrir en pérdidas es inmediato si el mercado se mueve en contra (para posiciones cortas).  
- En opciones at-the-money (S₀ ≈ K), la call y el put tienen primas similares por paridad put-call; cuando la opción está fuera del dinero (OTM), la prima es pequeña y el BE está muy cerca del strike.


### Pregunta 2  
**Si SPY termina exactamente en K al vencimiento, ¿cuánto gana o pierde cada posición? ¿Por qué?**

---

Si `S_T = K`, el payoff **intrínseco** de ambas opciones es **cero** porque:

- Call: `max(S_T − K, 0) = max(0, 0) = 0`  
- Put:  `max(K − S_T, 0) = max(0, 0) = 0`

Solo cuenta la prima pagada o cobrada:

| Posición | P&L en S_T = K | Cálculo |
|---|---|---|
| **Long Call** | **−C** | Compré el derecho, no ejercí; pierdo toda la prima |
| **Short Call** | **+C** | Vendí el derecho, no se ejerció; me quedo con toda la prima |
| **Long Put** | **−P** | Compré el derecho, no ejercí; pierdo toda la prima |
| **Short Put** | **+P** | Vendí el derecho, no se ejerció; me quedo con toda la prima |

**¿Por qué?**  
Al expirar exactamente en el strike, ninguna opción tiene valor intrínseco. El titular no tiene incentivo para ejercer (ganar $0) pero ya pagó la prima. El lanzador (short) se beneficia completamente: recibió el dinero y no tiene obligación de pago. Esto ilustra que el **valor temporal** de la opción cae a cero en el vencimiento.


### Pregunta 3  
**Compara la pérdida máxima de una long call vs. una short put. ¿Cuál tiene más riesgo? ¿Por qué?**

---

| Métrica | Long Call | Short Put |
|---|---|---|
| **Pérdida máxima** | **−C** (prima pagada, limitada) | **−(K − P)** si S→0 (muy grande) |
| **Ganancia máxima** | Ilimitada (↑ S_T) | **+P** (limitada a la prima cobrada) |
| **Perfil de riesgo** | Riesgo acotado | Riesgo muy elevado |

**La Short Put tiene mucho más riesgo**, y aquí está el por qué:

1. **Long Call:** El peor escenario es que SPY expire por debajo del strike y la opción venza sin valor. La pérdida está **perfectamente acotada** a la prima `C`. No importa cuánto caiga el mercado: el comprador del call ya perdió todo lo que podía perder desde el inicio.

2. **Short Put:** El vendedor del put asume la *obligación* de comprar SPY al precio K si el comprador ejerce. Si SPY cae drásticamente (imaginemos una crisis), la pérdida puede ser enorme: `−(K − P)` si el activo llegara a cero. En la práctica, si SPY cae 20-30 %, la pérdida puede superar muchas veces la prima cobrada.

**Conclusión:** La asimetría es fundamental en opciones. La long call limita el riesgo al capital invertido (la prima), mientras que la short put puede generar pérdidas que multiplican varias veces la prima cobrada si el mercado cae fuertemente.


### Pregunta 4  
**¿En qué escenario de mercado elegirías una long put sobre una short call, si ambas expresan una visión bajista?**

---

Ambas son estrategias bajistas, pero con perfiles de riesgo radicalmente distintos:

| Característica | **Long Put** | **Short Call** |
|---|---|---|
| Coste inicial | Pagas prima P | **Cobras** prima C |
| Pérdida máxima | P (limitada) | Ilimitada (S_T→∞) |
| Ganancia máxima | K − P (si S→0) | C (limitada) |
| Necesita que el activo caiga para ganar | Sí (y más allá del BE) | No (basta con que *no suba* más allá de K+C) |
| Se beneficia de subidas de volatilidad | ✅ Long vega | ❌ Short vega |

**Elegiría una Long Put en lugar de una Short Call cuando:**

1. **El movimiento bajista esperado es grande y/o rápido:** La long put tiene perfil convexo; cuanto más cae SPY, mayor es la ganancia. La short call solo captura la prima fija.

2. **La volatilidad implícita está baja y espero que suba:** La long put se beneficia de aumentos en volatilidad (vega positiva). Comprar opciones baratas y que luego suba la vol es una ventaja enorme.

3. **No tengo el subyacente en cartera:** Una short call "desnuda" (naked) es extraordinariamente peligrosa: si SPY sube un 20 %, la pérdida es ilimitada. Sin cobertura, la short call desnuda es inapropiada para la mayoría de inversores.

4. **Quiero un riesgo definido y conocido desde el inicio:** La long put actúa como un seguro: sé exactamente cuánto puedo perder (la prima P) antes de entrar en la operación.

5. **Uso la opción como cobertura (hedge) de una cartera larga:** La long put protege una cartera de acciones contra caídas, algo que la short call no logra.

**Cuándo preferir la Short Call:**  
Si poseo SPY (covered call), puedo vender calls para generar ingresos adicionales mientras acepto limitar mi ganancia si sube. También si la volatilidad implícita está muy alta (opciones caras) y espero que caiga — pero siempre con el riesgo de una subida explosiva.

**Resumen:** La long put es la herramienta bajista preferida cuando se busca **riesgo definido, potencial de ganancia elevado y protección ante movimientos abruptos**. La short call conviene principalmente como estrategia de generación de ingresos sobre posiciones ya existentes (covered call).


### **Pregunta 1**

¿Cuál de las cuatro posiciones básicas tiene el break-even más cercano al precio actual? ¿Qué implica eso?

El break-even más cercano a S₀ es el que necesita el menor movimiento del precio para empezar a generar ganancias.

En calls, el break-even es K + C, el precio tiene que subir lo suficiente para cubrir la prima.
En puts, es K − P, el precio tiene que bajar lo suficiente. Entonces, la comparación realmente es cuál de estos dos está más cerca del precio actual.

¿Qué significa esto en la práctica?

Si el break-even está muy cerca, la posición “entra en zona de ganancia” con un movimiento pequeño.
Pero también implica que, si estás vendido (short), puedes empezar a perder casi de inmediato si el mercado se mueve en tu contra.

En general, cuando las opciones están cerca del dinero (ATM), calls y puts tienen primas parecidas, así que sus break-even no suelen estar muy lejos uno del otro.

### **Pregunta 2**

Si SPY termina exactamente en K al vencimiento, ¿cuánto gana o pierde cada posición? ¿Por qué?

Si al vencimiento S_T = K, ninguna opción tiene valor intrínseco:

La call no vale nada porque no conviene comprar a K si el precio es K.
La put tampoco, por la misma razón.

Entonces, todo se reduce a la prima:

Long Call: pierde la prima C
Short Call: gana C
Long Put: pierde P
Short Put: gana P

¿Por qué pasa esto?
Porque las opciones expiran “sin valor”. El que compró pagó por un derecho que nunca usó, y el que vendió se queda con ese dinero. Es el ejemplo más claro de cómo el valor temporal desaparece al vencimiento.

### **Pregunta 3**

Compara la pérdida máxima de una long call vs. una short put. ¿Cuál tiene más riesgo? ¿Por qué?

La diferencia aquí es bastante importante:

Long Call:
Pierde como máximo la prima C
Ganancia potencial ilimitada
Short Put:
Gana como máximo la prima P
Puede perder mucho si el precio cae fuerte

En una long call, desde el inicio sabes exactamente cuánto puedes perder. No importa qué tan mal se comporte el mercado, nunca pierdes más que la prima.

En cambio, en una short put, estás obligado a comprar el activo a K si el precio cae. Si el mercado se desploma, la pérdida puede ser muy grande (en el extremo, si el precio se fuera a cero).

Conclusión: la short put tiene bastante más riesgo, porque sus pérdidas pueden crecer mucho más que la ganancia potencial.

### ****Pregunta 4**

¿En qué escenario elegirías una long put sobre una short call, si ambas expresan una visión bajista?

Aunque las dos son bajistas, no son equivalentes.

Elegiría una long put sobre una short call cuando:

Espero una caída fuerte o rápida del mercado (la put gana más entre más cae el precio).
La volatilidad está baja y puede subir (la long put se beneficia de eso).
No tengo el activo para cubrir una short call (evitar riesgo ilimitado).
Quiero saber desde el inicio cuál es mi pérdida máxima.
Busco protección (por ejemplo, cubrir una cartera de acciones).

La short call tiene más sentido en casos como:

Cuando ya tienes el activo (covered call).
Cuando solo quieres generar ingreso extra y no esperas grandes subidas.

En resumen:
La long put es más “segura” en términos de riesgo porque está acotado, mientras que la short call puede volverse muy peligrosa si el mercado sube fuerte.